# Sequence Prediction - N-BACK protocol

Copy of `_results_sequence_prediction_strict.ipynb`, but every stage runs with
`--nback K` instead of `--strict`. The n-back protocol predicts a screen from
**exactly `K` preceding screens**, sliding over the trace, so the target is
always the *end of the chain*: `K=1` predicts a screen from its immediate
predecessor, `K=2` predicts screen 3 from screens 1-2, and so on.

`--nback` is an alternative to `--strict`, not a variant of it. The two flags are
mutually exclusive: `--strict` must keep reproducing Screen2Vec's own protocol
(`modeltester.py`), where a screen in the middle of the trace is predicted.

| | permissive (default notebook) | `--strict` | `--nback K` (this notebook) |
|---|---|---|---|
| traces kept | >= 3 screens | >= `n_predictors + 1` (= 5) | >= `K + 1` |
| context per prediction | 1..4 screens (truncated near trace start) | exactly 4 | exactly `K` |
| target | next screen | next screen, full windows only | end of the `K`-chain, full windows only |
| candidate pool | every embedding in the directory | screens of eligible traces | screens of eligible traces |
| top-k% thresholds | `ceil()` | `int()` truncation (`modeltester.py:141`) | `ceil()` |
| training negatives | any screen | screens of eligible traces | screens of eligible traces |

Because every training window is full under `--nback`, the zero-padding branch in
the loss never fires, so training and evaluation see the same context length.
That mismatch is present in the permissive pipeline.

**Artefacts are suffixed `_nback{K}`** and cannot clobber a permissive or a
strict run (nor a run with a different `K`):

    embeddings_seq_768_nback{K}/                    (9_)
    lstm_predictor_768d_nback{K}.pth                (9_)
    sequence_projector_768d_nback{K}.pth            (9_)
    embeddings_seq_768_nback{K}_with_app_desc/      (6_)

`6_generate_app_context_embeddings.py` needs no flag of its own - it just
concatenates an app vector per screen, so it is pointed at the n-back
directories instead.

### Three caveats on the numbers this produces

1. **Numbers from different `K` are not directly comparable.** The eligible-trace
   population shrinks as `K` grows (>= `K + 1` screens), and the candidate pool -
   and therefore the top-k% thresholds and the retrieval difficulty - shrinks with
   it: ~64.4k pool screens at `K=1`, ~61.7k at `K=2`, ~54.5k at `K=4`. A change in
   accuracy across `K` mixes the effect of context length with the effect of an
   easier pool. Read each `K` on its own, or re-run with a fixed pool if you want
   a clean sweep.

2. **Screen ordering.** Screen2Vec builds each trace from an unsorted
   `os.listdir()` of `view_hierarchies`
   (`UI_embedding/precompute_embeddings.py:72`), which is not temporal order and
   is not reproducible on another filesystem. This pipeline always uses
   `gestures.json` order, i.e. true temporal order. That matters here because the
   n-back protocol is defined over the temporal chain.

3. **Screen-ID namespace bug, unresolved.** Embeddings are keyed by Rico's
   *global* `data/layouts/` numbering, while `gestures.json` keys are *per-trace*
   indices. They are different namespaces, so every trace is currently matched to
   unrelated screens (verified: 0 of 731 sampled trace screens match
   `layouts/<id>.json`). `--nback` does not fix this and inherits it. Treat the
   output below as protocol-comparable, not yet correct.

In [1]:
# ==== Configuration: set N here, everything below follows from it ====
NBACK = 1        # number of preceding screens used as context (>= 1)
PROJ_DIM = 768   # projector output dimension

EMB_DIR = f"embeddings_seq_{PROJ_DIM}_nback{NBACK}"
EMB_APP_DIR = f"{EMB_DIR}_with_app_desc"
LSTM_PATH = f"lstm_predictor_{PROJ_DIM}d_nback{NBACK}.pth"

print(f"n-back = {NBACK}  |  proj_dim = {PROJ_DIM}")
print(f"  projected embeddings : {EMB_DIR}/")
print(f"  + app context        : {EMB_APP_DIR}/")
print(f"  LSTM checkpoint      : {LSTM_PATH}")

n-back = 1  |  proj_dim = 768
  projected embeddings : embeddings_seq_768_nback1/
  + app context        : embeddings_seq_768_nback1_with_app_desc/
  LSTM checkpoint      : lstm_predictor_768d_nback1.pth


In [2]:
%run 9_train_joint.py --proj_dim $PROJ_DIM --nback $NBACK

Device: cpu
Joint training: Projector (4096 → 1024 → 768) + LSTM (768)
Loading 66261 embeddings from 'embeddings_4096_last' in parallel...


Loading embeddings: 100%|██████████| 66261/66261 [00:38<00:00, 1735.24it/s]


  Loaded matrix shape: (66261, 4096)
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')
Parsing 10292 traces in parallel...


Parsing traces: 100%|██████████| 10292/10292 [00:02<00:00, 4366.49it/s]


  Parsed 8527 traces
  NBACK (K=1): traces >= 2 screens, negative pool = 64408 of 66261 screens
  Input dim: 4096  |  Traces: 8527  |  Screens: 66261
  Train traces: 7675  |  Val traces: 852
  Projector parameters: 4,980,736
  LSTM parameters:      4,724,736
  Total parameters:     9,705,472

  Joint Training: 120 epochs, batch_size=256
  neg_samp=128, n_predictors=1
  temperature=0.07
  trace_negatives=context

Epoch   1/120 | Train: 4.5724 | Val: 3.0141 * best
Epoch   2/120 | Train: 2.9021 | Val: 2.6307 * best
Epoch   3/120 | Train: 2.5658 | Val: 2.5347 * best
Epoch   4/120 | Train: 2.3124 | Val: 2.3566 * best
Epoch   5/120 | Train: 2.1977 | Val: 2.3196 * best
Epoch   6/120 | Train: 2.0050 | Val: 2.2567 * best
Epoch   7/120 | Train: 1.9076 | Val: 2.3776 (patience 1/15)
Epoch   8/120 | Train: 1.7865 | Val: 2.3017 (patience 2/15)
Epoch   9/120 | Train: 1.6802 | Val: 2.2090 * best
Epoch  10/120 | Train: 1.6201 | Val: 2.2049 * best
Epoch  11/120 | Train: 1.5190 | Val: 2.1638 * best
Epoch

Projecting: 100%|██████████| 17/17 [00:02<00:00,  7.31it/s]


  Projected shape: (66261, 768)


Saving embeddings: 100%|██████████| 66261/66261 [00:57<00:00, 1144.02it/s]


Saved LSTM predictor: lstm_predictor_768d_nback1.pth
Saved projector: sequence_projector_768d_nback1.pth
Saved training plot: joint_training_768d_nback1.png

Total training time: 9516.1s

Next steps:
  python 6_generate_app_context_embeddings.py \
    --embeddings_dir embeddings_seq_768_nback1 \
    --output_dir embeddings_seq_768_nback1_with_app

  python 4_evaluate_screen_prediction.py \
    --embeddings_dir embeddings_seq_768_nback1_with_app \
    --lstm_model lstm_predictor_768d_nback1.pth \
    --nback 1


In [3]:
%run 6_generate_app_context_embeddings.py --embeddings_dir $EMB_DIR --output_dir $EMB_APP_DIR

C:\Users\Jonathan\Dev\screen2vec-revisited\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading app details from ../rico-data/data/play_store_metadata/app_details_enriched.csv...
  CSV columns: ['App Package Name', 'Play Store Name', 'Category', 'Average Rating', 'Number of Ratings', 'Number of Downloads', 'Date Updated', 'Icon URL', 'gp_title', 'gp_description', 'gp_descriptionHTML', 'gp_summary', 'gp_installs', 'gp_minInstalls', 'gp_realInstalls', 'gp_score', 'gp_ratings', 'gp_reviews', 'gp_histogram', 'gp_price', 'gp_free', 'gp_currency', 'gp_sale', 'gp_saleTime', 'gp_originalPrice', 'gp_saleText', 'gp_offersIAP', 'gp_inAppProductPrice', 'gp_developer', 'gp_developerId', 'gp_developerEmail', 'gp_developerWebsite', 'gp_developerAddress', 'gp_privacyPolicy', 'gp_genre', 'gp_genreId', 'gp_categories', 'gp_icon', 'gp_headerImage', 'gp_screenshots', 'gp_video', 'gp_videoImage', 'gp_contentRating', 'gp_contentRatingDescription', 'gp_adSupported', 'gp_containsAds', 'gp_released', 'gp_lastUpdatedOn', 'gp_updated', 'gp_version', 'gp_comments', 'gp_status']
  Found description c

Parsing layout JSONs: 100%|██████████| 104/104 [00:00<00:00, 8580.01it/s]

Preparing text input for embedding model...
Loading SBERT model 'sentence-transformers/all-mpnet-base-v2' (embed_dim=768)...



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6772.86it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding app packages...


... (2 outputs left out)

Saving concatenated embeddings:   0%|          | 111/66261 [00:00<01:58, 556.00it/s]

Concatenating screen (768-dim) + app (768-dim) embeddings...
Output dimension: 1536-dim


Saving concatenated embeddings: 100%|██████████| 66261/66261 [01:35<00:00, 690.49it/s]


Success! Concatenated embeddings created and saved.


In [ ]:
%run 4_evaluate_screen_prediction.py --embeddings_dir $EMB_APP_DIR --lstm_model $LSTM_PATH --nback $NBACK


[Step 0/3] Loading LSTM predictor from 'lstm_predictor_768d_nback1.pth'...
  Loaded LSTM predictor (embed_dim=768, best_val_loss=2.0063)

[Step 1/3] Loading embeddings...
Loading 66261 embeddings from 'embeddings_seq_768_nback1_with_app_desc' in parallel...


Loading embeddings: 100%|██████████| 66261/66261 [00:30<00:00, 2153.14it/s]


  Loaded matrix shape: (66261, 1536)
  66261 screen embeddings available

[Step 2/3] Parsing RICO interaction traces...
  Loaded trace->layout index: 66157 screens ('trace_layout_index.csv')


Parsing traces: 100%|██████████| 9384/9384 [00:37<00:00, 251.77it/s] 


  Parsed 8527 traces (64408 screens)
  Skipped 1765 traces with < 2 screens
  Skipped 104 screens absent from the trace->layout index
  N-back (K=1) pool: restricted to 64408 screens from eligible traces (was 66261)

[Step 3/3] Running screen prediction evaluation...

  Protocol: N-BACK (predicting the end of the chain from exactly 1 preceding screens)

  Total screens in embedding index: 64408
  Top-0.01% threshold: k=7
  Top-0.1%  threshold: k=65
  Top-1%    threshold: k=645
  Top-5%    threshold: k=3221
  Mean embedding L2 norm: 1.4142
  Prediction mode: LSTM (Screen2Vec-equivalent)
  Evaluating traces in parallel via threads (n_jobs=-1)...


Evaluating traces (LSTM):  93%|█████████▎| 7896/8527 [1:13:53<06:52,  1.53it/s]